In [ ]:
import copy
import os

import numpy as np
from barrier3d import Barrier3d
from matplotlib import pyplot as plt

from cascade.outwasher2 import Outwasher

In [ ]:
b3d = Barrier3d.from_yaml("C:/Users/Lexi/PycharmProjects/Barrier3d/tests/test_params/")
years = np.load(
    "C:/Users/Lexi/PycharmProjects/CASCADE/cascade/data/outwash_data/outwash_years.npy"
)
print(years)

for t in range(1, 64):
    b3d.update()
    b3d.update_dune_domain()
    print("B3D time step: ", b3d._time_index)

outwash = Outwasher(
    datadir="C:/Users/Lexi/PycharmProjects/CASCADE/cascade/data/outwash_data/",
    outwash_years="outwash_years.npy",
    outwash_bay_levels="outwash_baylevels10.npy",
    time_step_count=b3d._TMAX,
    berm_elev=b3d._BermEl,
    barrier_length=b3d._BarrierLength,
    sea_level=b3d._SL,
    bay_depth=-b3d._BayDepth,
    interior_domain=b3d.InteriorDomain,
    dune_domain=b3d.DuneDomain[b3d._time_index - 1],
    block_size=5,
    substep=600,
    sediment_flux_coefficient_Cx=10,
    sediment_flux_coefficient_Ki=7.5e-3,  # b3d = 7.5E-6 for inundation
    max_slope=-0.25,
    shoreface_on=True,
)
outwash.update(b3d)

In [ ]:
outwash._Qs_shoreface[b3d._time_index - 1]

In [ ]:
path = "D:/NC State/Outwasher/Output/newest_flow_routing/"
runID = "100years_outwash10_substep20"
newpath = path + runID + "/"
if not os.path.exists(newpath):
    os.makedirs(newpath)
os.chdir(newpath)

plt.rcParams["figure.figsize"] = (8, 6)
plt.rcParams.update({"font.size": 15})

initial_domain = outwash._initial_full_domain
final_domain = outwash._full_domain
domain_change = final_domain - initial_domain

### ELEVATION PLOTS-------------------------------------------------------------------------------------------------------------
# plotting initial domain
fig1 = plt.figure()
ax1 = fig1.add_subplot(111)
mat = ax1.matshow(
    initial_domain * 10,
    cmap="terrain",
    vmin=-3.0,
    vmax=3.0,
)
cbar = fig1.colorbar(mat)
cbar.set_label("m MHW", rotation=270, labelpad=15)
ax1.set_title("Initial Elevation")
ax1.set_ylabel("barrier width (dam)")
ax1.set_xlabel("barrier length (dam)")
plt.gca().xaxis.tick_bottom()
# fig1.savefig(newpath + "0_domain", facecolor='w')

# plotting post storm elevation
fig2 = plt.figure()
ax2 = fig2.add_subplot(111)
mat2 = ax2.matshow(
    final_domain[:, :] * 10,
    cmap="terrain",
    vmin=-3.0,
    vmax=3.0,
)
ax2.set_xlabel("barrier length (dam)")
ax2.set_ylabel("barrier width (dam)")
ax2.set_title("Final Elevation")
plt.gca().xaxis.tick_bottom()
cbar = fig2.colorbar(mat2)
cbar.set_label("m MHW", rotation=270, labelpad=15)
# fig2.savefig(newpath + "final_domain", facecolor='w')

# plotting domain elevation change
domain_change = final_domain - initial_domain
fig3 = plt.figure()
ax3 = fig3.add_subplot(111)
mat3 = ax3.matshow(
    domain_change * 10,
    cmap="seismic",
    vmin=-2,
    vmax=2,
)
ax3.set_xlabel("barrier length (dam)")
ax3.set_ylabel("barrier width (dam)")
ax3.set_title("Elevation Change")
plt.gca().xaxis.tick_bottom()
cbar = fig3.colorbar(mat3)
cbar.set_label("meters", rotation=270, labelpad=15)
# fig3.savefig(newpath + "elev_change_domain", facecolor='w')